# 05 — Phase 2: Rank Selection  ⚡ LIVE — run this in the demo

Pure math on the stored Phase 1 results — no model evaluation, near-instant.
For each layer it:

1. **inverts** the biquadratic polynomial to find the max noise tolerable
   under a global mAP budget,
2. builds the **Pareto front** over (compression ratio, mAP),
3. **K-means clusters** the front into Conservative / Balanced / Aggressive
   suggestions.

You then pick one tier, apply the decomposition, recalibrate BatchNorm, and
save the compressed model. This is the identical Phase 2 logic as the CNN
pipeline, with mAP as the quality axis.

In [9]:
import sys, os, json, glob
sys.path.append(os.path.abspath("../src"))
import numpy as np
import torch
from torch.utils.data import DataLoader

from model import YOLOv3
from dataset import CocoSubsetDataset, yolo_collate_fn
from tucker_pipeline import (select_ranks_for_layer, apply_selected_ranks,
                              recalibrate_batchnorm, get_module_by_name)

with open("../checkpoints/run_config.json") as f:
    cfg = json.load(f)
CLASS_NAMES, IMG_SIZE, DATA_ROOT, BATCH_SIZE = cfg["class_names"], cfg["img_size"], cfg["data_root"], cfg["batch_size"]
NUM_CLASSES = len(CLASS_NAMES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


## Load stored Phase 1 results

In [10]:
layer_results = []
for path in sorted(glob.glob("../checkpoints/phase1/phase1_layer*.json")):
    with open(path) as f:
        layer_results.append(json.load(f))
with open("../checkpoints/phase1/baseline.json") as f:
    baseline_map = json.load(f)["baseline_map"]

print(f"loaded {len(layer_results)} layers, baseline mAP@0.5 = {baseline_map:.4f}")

loaded 37 layers, baseline mAP@0.5 = 0.0402


## Set the global budget and run selection

`BUDGET_PCT` is a **percentage of baseline mAP** you're willing to give up
(e.g. 5.0 = allow the fitted curve to drop to 95% of baseline mAP). Change
this live to show tighter vs looser compression.

In [11]:
BUDGET_PCT = 2.0
quality_budget = baseline_map * (BUDGET_PCT / 100.0)   # absolute mAP drop allowed
print(f"budget: {BUDGET_PCT}% of baseline -> allow up to {quality_budget:.4f} mAP drop")

selections_all = []
for lr in layer_results:
    if lr["poly_coeffs"] is None:
        selections_all.append({"layer": lr["name"], "max_noise": None, "suggestions": []})
        continue
    # baseline reference on this layer's own curve = poly value at noise=0
    layer_baseline = float(np.poly1d(lr["poly_coeffs"])(0))
    sel = select_ranks_for_layer(lr["name"], lr["sweep_results"], lr["poly_coeffs"],
                                  layer_baseline, quality_budget)
    selections_all.append(sel)

n_with = sum(1 for s in selections_all if s["suggestions"])
print(f"{n_with}/{len(selections_all)} layers have suggestions within budget")

budget: 2.0% of baseline -> allow up to 0.0008 mAP drop
37/37 layers have suggestions within budget


## Inspect suggestions for a few layers

In [12]:
for sel in selections_all[:5]:
    if not sel["suggestions"]:
        print(f"{sel['layer']}: (no suggestion within budget)")
        continue
    print(f"\n{sel['layer']}  (max tolerable noise {sel['max_noise']:.1f}%)")
    for s in sel["suggestions"]:
        print(f"  {s['label']:13s} r_in={s['r_in']:4d} r_out={s['r_out']:4d} "
              f"noise={s['noise_percent']:5.1f}% mAP={s['mAP']:.4f} CR={s['compression_ratio']:.2f}x")


backbone.stage1.0.0  (max tolerable noise 15.8%)
  Conservative  r_in=  30 r_out=  60 noise= 10.6% mAP=0.0425 CR=0.88x
  Balanced      r_in=  32 r_out=  54 noise= 14.9% mAP=0.0414 CR=0.92x

backbone.stage1.1.conv2.0  (max tolerable noise 55.2%)
  Conservative  r_in=  30 r_out=  60 noise= 14.9% mAP=0.0417 CR=0.88x
  Conservative  r_in=  18 r_out=  64 noise= 34.8% mAP=0.0416 CR=1.23x
  Conservative  r_in=  18 r_out=  60 noise= 36.0% mAP=0.0415 CR=1.30x
  Balanced      r_in=  12 r_out=  64 noise= 48.3% mAP=0.0410 CR=1.62x
  Balanced      r_in=  12 r_out=  60 noise= 48.5% mAP=0.0400 CR=1.72x
  Aggressive    r_in=  12 r_out=  42 noise= 53.1% mAP=0.0400 CR=2.42x
  Aggressive    r_in=  12 r_out=  36 noise= 54.9% mAP=0.0385 CR=2.80x

backbone.stage2.0.0  (max tolerable noise 14.6%)
  Conservative  r_in=  64 r_out= 117 noise= 11.1% mAP=0.0419 CR=0.85x

backbone.stage2.1.conv2.0  (max tolerable noise 33.5%)
  Conservative  r_in=  39 r_out= 104 noise= 28.2% mAP=0.0411 CR=1.41x
  Balanced      r_

## Pick a tier and build the per-layer rank map

Choose one tier globally (`Conservative` / `Balanced` / `Aggressive`). For
each layer we take its suggestion at that tier (falling back to the closest
available if that layer's front has fewer than 3 clusters).

In [5]:
TIER = "Balanced"

selected_ranks = {}
for sel in selections_all:
    if not sel["suggestions"]:
        continue
    match = [s for s in sel["suggestions"] if s["label"] == TIER]
    chosen = match[0] if match else sel["suggestions"][len(sel["suggestions"]) // 2]
    selected_ranks[sel["layer"]] = {"r_in": chosen["r_in"], "r_out": chosen["r_out"]}

print(f"selected ranks for {len(selected_ranks)} layers at tier '{TIER}'")

selected ranks for 37 layers at tier 'Balanced'


## Apply decomposition to the model

In [6]:
model = YOLOv3(num_classes=NUM_CLASSES)
state = torch.load("../checkpoints/yolov3_best.pt", map_location=DEVICE)
model.load_state_dict(state["model_state"])
model.to(DEVICE).eval()

from analysis_utils import count_params
p_before = count_params(model)
model = apply_selected_ranks(model, selected_ranks, DEVICE)
p_after = count_params(model)
print(f"params: {p_before:,} -> {p_after:,}  ({p_after/p_before:.1%} kept)")

/tmp/ipykernel_1315/2306084478.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load("../checkpoints/yolov3_best.pt", map_location=DEVICE)


params: 61,545,274 -> 11,481,767  (18.7% kept)


## Recalibrate BatchNorm

Essential after decomposition — resyncs BN running stats to the factored
layers' outputs (forward-only, no weight changes). Without this the
compressed model produces erratic, image-dependent detection counts.

In [7]:
recal_ds = CocoSubsetDataset(DATA_ROOT, "train2017", CLASS_NAMES, img_size=IMG_SIZE,
                              images_per_class=30, augment=False)
recal_loader = DataLoader(recal_ds, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=yolo_collate_fn, num_workers=0)
model = recalibrate_batchnorm(model, recal_loader, DEVICE, num_batches=20)
print("BatchNorm recalibrated")

loading annotations into memory...
Done (t=10.58s)
creating index...
index created!
BatchNorm recalibrated


## Save the compressed model

In [8]:
torch.save(model.state_dict(), "../checkpoints/yolov3_compressed.pt")
with open("../checkpoints/compression_selection.json", "w") as f:
    json.dump({"tier": TIER, "budget_pct": BUDGET_PCT, "selected_ranks": selected_ranks}, f, indent=2)
print("saved ../checkpoints/yolov3_compressed.pt")

x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    outs = model(x)
print("output shapes:", [tuple(o.shape) for o in outs])

saved ../checkpoints/yolov3_compressed.pt
output shapes: [(1, 30, 13, 13), (1, 30, 26, 26), (1, 30, 52, 52)]
